# 用尽量少的硬币付款：用 Optimize 直接求出最省的那一组

## 这一章要分清的两类问题

Z3 能够回答的问题分为两类，其求解方式并不相同。

- 约束满足问题：找到哪怕一个满足条件的解
- 最优化问题：合格的解往往有一大堆，要从其中挑出最好的那一个

同一个“硬币枚数”的题目，这一章会用两种办法各做一遍，看差别在哪里。

- 用 `Solver` 把解逐个求出来，自己比较，除了一枚一枚地数没有别的办法
- 用 `Optimize` 直接要“枚数最少的那个解”
- 用 `lower()` / `upper()` 确认这确实是最好的，而不是碰巧遇到

## 先给出题目

134 美分要付掉，如何搭配硬币，用出去的枚数才能尽量少。

- 手上有这些硬币：1, 2, 5, 10, 25, 50 美分
- 要凑的金额：134 美分
- 目的：用出去的硬币枚数越少越好

“合计正好 134 美分”是约束，“枚数尽量小”是目的，两者的性质并不相同。在 Z3 中，这两件事必须分开写，这一章的关键就在这里。

## 方法一：用 Solver 求出全部合格的解

`Solver` 做的是找出满足条件的解这件事。我们想知道最少的枚数究竟是多少，因此这一次干脆把合格的解全部取出来，枚数自己一个一个比较。

In [1]:
from z3 import *
from helper.z3 import all_solutions   # 本书准备的“依次返回全部解”函数

coins = [1, 2, 5, 10, 25, 50]   # 可用的硬币种类（美分）
total = 134                     # 要凑的金额
max_coins = 8                   # 先限定在“8 枚以下”

counts = [Int(f"n_{c}") for c in coins]                # 每种硬币的枚数
amount = Sum([c * n for c, n in zip(coins, counts)])   # 合计金额
num_coins = Sum(counts)                               # 合计枚数

solver = Solver()
for n in counts:
    solver.add(n >= 0)              # 枚数在 0 以上
solver.add(amount == total)         # 合计正好 134 美分
solver.add(num_coins <= max_coins)  # 枚数在 8 枚以下


def show(model):
    """把解显示成“8 枚: 2 美分×2, 5 美分×1, ...”的形式"""
    used = {c: model.eval(n).as_long() for c, n in zip(coins, counts)}
    used = {c: k for c, k in used.items() if k > 0}   # 省略 0 枚的硬币
    print(f"{model.eval(num_coins)} 枚: " + ", ".join(f"{c} 美分×{k}" for c, k in used.items()))


for model in all_solutions(solver):
    show(model)

8 枚: 2 美分×2, 5 美分×1, 25 美分×5
8 枚: 1 美分×2, 2 美分×1, 10 美分×3, 50 美分×2
8 枚: 1 美分×1, 2 美分×4, 25 美分×1, 50 美分×2
8 枚: 1 美分×4, 5 美分×1, 25 美分×1, 50 美分×2
7 枚: 2 美分×2, 5 美分×1, 25 美分×3, 50 美分×1
8 枚: 2 美分×2, 10 美分×3, 25 美分×2, 50 美分×1
8 枚: 1 美分×2, 2 美分×1, 5 美分×1, 25 美分×3, 50 美分×1
7 枚: 2 美分×2, 10 美分×3, 50 美分×2
8 枚: 2 美分×2, 5 美分×2, 10 美分×2, 50 美分×2
7 枚: 1 美分×2, 2 美分×1, 5 美分×1, 25 美分×1, 50 美分×2
6 枚: 2 美分×2, 5 美分×1, 25 美分×1, 50 美分×2


合格的解一共找到 11 种，其中最省的是 6 枚的那种搭配。

可是这条路有它的短板，下面三条都值得注意。

- `Solver` 并不按枚数从少到多的顺序去找，这一次 6 枚那个是最后才出现的，下一次能否排在前面并没有保证
- 得先攒齐所有解再比较，解一多就慢，事实上把 `max_coins` 的上限一撤，解就有 11,713 种
- 8 枚这个上限也是我们随手定的，谁也没有证明过 6 枚以下付不出来

要找最好的那个解，需要换一件工具，即 `Optimize`，它专门处理这类问题。

## 方法二：用 Optimize 直接求“枚数最少的解”

`Optimize` 做的事，是在一堆合格解里挑出让目标函数(这里就是枚数)最小或最大的那一个。写法与 `Solver` 几乎一样，只是最后多出一行 `minimize()`。

In [2]:
opt = Optimize()

for n in counts:
    opt.add(n >= 0)
opt.add(amount == total)        # 金额条件相同。枚数上限已经不需要了

obj = opt.minimize(num_coins)   # 目标函数：让合计枚数最小

print(opt.check())

model = opt.model()
show(model)
print("合计:", sum(c * model.eval(n).as_long() for c, n in zip(coins, counts)), "美分")

sat
6 枚: 2 美分×2, 5 美分×1, 25 美分×1, 50 美分×2
合计: 134 美分


答案还是那 6 枚，与前面逐个比较的结果相同。顺带留意 `minimize()` 的返回值：它给出的是目标函数的一个句柄，拿着这个句柄，才能去问最好能好到什么程度，以及这个结论靠不靠得住。

In [3]:
print("最优值的下界:", opt.lower(obj))
print("最优值的上界:", opt.upper(obj))

# 把条件改差一点比较看看（加上 7 枚以上，就选不到 6 枚的解）
opt.add(num_coins >= 7)
opt.check()
print("在 7 枚以上的条件下:", opt.lower(obj), opt.upper(obj))

最优值的下界: 6
最优值的上界: 6
在 7 枚以上的条件下: 7 7


最优值的边界就是一句保证：它落在这段范围里，即 `lower()` 与 `upper()` 的返回值。第一个结果两者都是 6，那就等于 Z3 把话说清楚了，即比 6 枚更少的付法根本不存在；第二种做法往条件里又加了一句，区间就跟着滑到 7。由于条件已经改变，因此 `Optimize` 会重新去找它的最优值。

只靠 `Solver`，最多能说“我找到的这些里它最好”；`Optimize` 则敢说“它就是最好”，两者的分水岭就在这里。

| | `Solver` | `Optimize` |
|---|---|---|
| 目的 | 找出一个满足条件的解 | 找出目标函数最优的解 |
| 添加约束 | `solver.add()` | `opt.add()` |
| 最优化 | 做不到 | `opt.minimize()` / `opt.maximize()` |
| 得到的答案 | 任意一个(不一定最优) | 最优的那一个 |
| 最优的保证 | 没有 | 可以用 `lower()` / `upper()` 确认 |

## 包成函数，换一套硬币再试

形状一样的问题，写成函数就能反复使用，下面也拿日元试一次。

In [4]:
def min_coin_counts(coin_types, amount):
    """返回用最少枚数凑出金额 amount 的组合"""
    counts = [Int(f"n_{c}") for c in coin_types]
    opt = Optimize()
    for n in counts:
        opt.add(n >= 0)
    opt.add(Sum([c * n for c, n in zip(coin_types, counts)]) == amount)
    opt.minimize(Sum(counts))
    assert opt.check() == sat
    model = opt.model()
    return {c: model[n].as_long() for c, n in zip(coin_types, counts) if model[n].as_long() > 0}


print(min_coin_counts([1, 2, 5, 10, 25, 50], 134))
print(min_coin_counts([1, 5, 10, 50, 100, 500], 1340))

{2: 2, 5: 1, 25: 1, 50: 2}
{10: 4, 100: 3, 500: 2}


硬币种类与金额一换，同一段代码照用不误，1340 日元跑出来是 500 日元×2、100 日元×3、10 日元×4，共 9 枚，正是枚数最少的那一组。

## 小结

- 约束满足问题问的是有没有合格解，最优化问题问的是合格的里面哪一个最好
- `Solver` 能给出一个解，也能把全部解都倒出来，可它不知道哪个最优
- `Optimize` 用 `minimize()` / `maximize()` 点明目标函数，再去找最优的那一个
- `lower()` 与 `upper()` 一碰头，那个数就是真正的最优值
- 靠“全部收起来再比较”的办法，解一多就不行了，这题把上限一撤就是 11,713 种，比较的代价随之暴涨